<a href="https://colab.research.google.com/github/lawesworks/vision_workbench/blob/main/notebooks/YOLO_OD_Trainer_Turnkey_CE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyfiglet
from pyfiglet import figlet_format

In [ ]:
# Mandatory
Project_Title              = None
Roboflow_Project_URL       = None
Roboflow_Dataset_Version   = None
# Optional
YOLO_Version               = None
YOLO_Size                  = None
Config_Epochs              = None
Config_Batches             = None
Config_Learning_Rate       = None
Config_Image_Size          = None

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Paste Configuration File contents here
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

Project_Title = "Satellite-Airplane"
Roboflow_Project_URL = "https://universe.roboflow.com/gxust/airplane-ksct2"
Roboflow_Dataset_Version = 1
YOLO_Version = "yolov8"
YOLO_Size = "s"
Config_Epochs = 30
Config_Batches = 16
Config_Learning_Rate = 0.01
Config_Image_Size = 640



In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Load Roboflow API Key from UserData / Secret Keys
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

from google.colab import userdata
Roboflow_API_Key = userdata.get('ROBOFLOW_API_KEY')

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Modify these fallback (default) values ony if needed
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

Project_Title               = Project_Title if Project_Title is not None else None
Roboflow_Project_URL        = Roboflow_Project_URL if Roboflow_Project_URL is not None else None

Roboflow_Dataset_Version    = Roboflow_Dataset_Version if Roboflow_Dataset_Version is not None else 1
YOLO_Version                = YOLO_Version if YOLO_Version is not None else "yolov8"
YOLO_Size                   = YOLO_Size if YOLO_Size is not None else "s"
Config_Epochs               = Config_Epochs if Config_Epochs is not None else 30
Config_Batches              = Config_Batches if Config_Batches is not None else 16
Config_Image_Size           = Config_Image_Size if Config_Image_Size is not None else 640
Config_Learning_Rate        = Config_Learning_Rate if Config_Learning_Rate is not None else 0.01
Roboflow_API_Key            = Roboflow_API_Key if Roboflow_API_Key is not None else None

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# CONFIGURATION CONSTANTS

# You do not need to change anything below this line
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

# URL Regex
#-------------------------------------------------------------------------------
import re
Roboflow_URL_Regex_Pattern = r"roboflow\.com/([^/]+)/([^/?#]+)"
match = re.search(Roboflow_URL_Regex_Pattern, Roboflow_Project_URL)

# Model Architectural Characteristics
#-------------------------------------------------------------------------------
DOWNLOAD_FORMAT                 = YOLO_Version

# Paths
#-------------------------------------------------------------------------------
DOWNLOAD_DIR                    = "./datasets"
LATEST_PREDICT_DIR              = "runs/detect/predict"
LATEST_TRAIN_DIR                = "runs/detect/train"

# Auto-Derive Project Information
#-------------------------------------------------------------------------------
Roboflow_Project_Name           = None
Roboflow_Project_Folder         = None
if match:
    Roboflow_Project_Workspace  = match.group(1)
    Roboflow_Project_Slug       = match.group(2)
else:
    print("Roboflow URL format not recognized")
    Roboflow_Project_Workspace  = None
    Roboflow_Project_Slug       = None

if (Project_Title is None):
  Project_Title = f"autotitle_{Roboflow_Project_Slug}"

# Minimum number of images in a dataset to qualify for training
#-------------------------------------------------------------------------------
MIN_TRAIN_IMAGES = 300

# Desired local split if dataset qualifies
#-------------------------------------------------------------------------------
TRAIN_RATIO = 0.70
VAL_RATIO = 0.2
TEST_RATIO = 0.1

# Random Seed - Starting Point
#-------------------------------------------------------------------------------
RANDOM_SEED = 42

# GPU Characteristics
#-------------------------------------------------------------------------------
Config_GPU_Batches_Per_Second = 3.5 # Iterations (Batches) per Second (As I Observed on Google TP4 GPU)

print(f"""
===== Training Configuration =====

API Key          : API Key Found

Project Title    : {Project_Title}

Project URL      : {Roboflow_Project_URL}
Workspace        : {Roboflow_Project_Workspace}
Project Slug     : {Roboflow_Project_Slug}
Dataset Version  : {Roboflow_Dataset_Version}

Project Name     : {Roboflow_Project_Name}
Project Folder   : {Roboflow_Project_Folder}
Download Dir     : {DOWNLOAD_DIR}

Download Format  : {DOWNLOAD_FORMAT}


Min Train Images : {MIN_TRAIN_IMAGES}
Training Ratio   : {TRAIN_RATIO}
Validation Ratio : {VAL_RATIO}
Test Ratio       : {TEST_RATIO}
Random Seed      : {RANDOM_SEED}

Model Framework  : {YOLO_Version}
Model Size       : {YOLO_Size}
Epochs           : {Config_Epochs}
Image Size       : {Config_Image_Size}
Batch Size       : {Config_Batches}

==================================
""")

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Load Libraries

# These are commonly used libraries that we will use in this script
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

#from IPython.display import Image, display
#from IPython.display import Image as DisplayImage, display
from pathlib import Path
import glob
import os
import math
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Image as DisplayImage, display
import torch

import tempfile
import yaml
import random
import shutil
import requests

from google.colab import files
from google.colab import drive

print(f"""
======= Imported Libraries =======

Ipython.display  : Image
Ipython.display  : display
pathlib          : Path
Ipython.display  : Image
Ipython.display  : DisplayImage
matplotlib       : plt
google.colab     : files
google.colab     : drive
glob
os
torch
random
shutil
requests
math
==================================
""")

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Mout Google Drive ...  The script will automatically save the best.pt weights file to it
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

!rm -rf /content/drive

#from google.colab import drive
drive.mount('/content/drive', force_remount=True)

print(Path("/content/drive/MyDrive").exists())
!ls /content/drive/MyDrive

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# GENERAL HELPER FUNCTIONS

# Reusable functions used in this script to automate repetitive or iterative steps / more to come
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


# function to copy files in the runtime to your mounted runtime drive (not very useful now)
# ----------------------------------------------------------------------------------------------------
def save_file_to_drive(input_file: str, destination: str):
    from pathlib import Path
    import shutil

    src = Path(input_file)
    dst = Path(destination)

    if not src.exists():
        raise FileNotFoundError(f"Input file not found: {src}")

    # If destination has a file extension → treat as file
    if dst.suffix:
        dst.parent.mkdir(parents=True, exist_ok=True)
        final_path = dst
    else:
        dst.mkdir(parents=True, exist_ok=True)
        final_path = dst / src.name

    shutil.copy(src, final_path)

    return str(final_path)


# function to copy files in the runtime to your google drive (refuses, if the drive is ot mounted)
# ----------------------------------------------------------------------------------------------------
def save_file_to_google_drive(input_file: str, destination: str):
    src = Path(input_file)
    dst = Path(destination)

    drive_root = Path("/content/drive/MyDrive")

    if not drive_root.exists():
        raise RuntimeError(
            "Google Drive is not mounted. Mount it first with drive.mount('/content/drive')."
        )

    if not src.exists():
        raise FileNotFoundError(f"Input file not found: {src}")

    if destination.endswith("/") or dst.suffix == "":
        dst.mkdir(parents=True, exist_ok=True)
        final_path = dst / src.name
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        final_path = dst

    shutil.copy2(src, final_path)
    return str(final_path)


# function to get GPU Summary
# ----------------------------------------------------------------------------------------------------


def get_gpu_cv_summary(device_index: int = 0) -> dict:
    """
    Return a concise GPU capability summary for CV / YOLO-style workloads.

    Returns a dict with:
      - property
      - value
      - why_it_matters
    """
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. Enable a GPU runtime or run on a CUDA-capable machine.")

    props = torch.cuda.get_device_properties(device_index)

    # Some attributes may not exist on all builds/versions; use getattr with fallback.
    summary = [
        {
            "property": "name",
            "value": props.name,
            "why_it_matters": "GPU generation & capabilities",
        },
        {
            "property": "total_memory",
            "value": f"{props.total_memory / (1024**3):.2f} GB",
            "why_it_matters": "Max batch size & image resolution",
        },
        {
            "property": "multi_processor_count",
            "value": props.multi_processor_count,
            "why_it_matters": "Parallel throughput",
        },
        {
            "property": "clock_rate",
            "value": f"{getattr(props, 'clock_rate', None) / 1000:.0f} MHz"
                     if getattr(props, "clock_rate", None) is not None else "N/A",
            "why_it_matters": "Kernel execution speed",
        },
        {
            "property": "memory_bus_width",
            "value": getattr(props, "memory_bus_width", "N/A"),
            "why_it_matters": "Data movement speed",
        },
        {
            "property": "warp_size",
            "value": props.warp_size,
            "why_it_matters": "Kernel efficiency",
        },
        {
            "property": "(major.minor)",
            "value": f"{props.major}.{props.minor}",
            "why_it_matters": "CUDA feature support / compute_capability ",
        },
    ]

    return {"device_index": device_index, "gpu_summary": summary}



# function to create image grid (Left → Right, Row Wrap) of images saved into the Predicted Folder
# ----------------------------------------------------------------------------------------------------
def show_image_grid_paged(image_paths, cols=5, per_page=20, page=1, figsize_per_cell=3):
    """
    Display images in a true grid, paged.
    - cols: images per row
    - per_page: total images per page
    - page: 1-based page index
    - figsize_per_cell: size multiplier per grid cell
    """

    if not image_paths:
        print("No images to display.")
        return

    start = (page - 1) * per_page
    end = min(start + per_page, len(image_paths))
    page_paths = image_paths[start:end]

    rows = math.ceil(len(page_paths) / cols)
    fig_w = cols * figsize_per_cell
    fig_h = rows * figsize_per_cell

    fig, axes = plt.subplots(rows, cols, figsize=(fig_w, fig_h))
    axes = axes.flatten() if isinstance(axes, (list, tuple)) is False else axes

    # If only one subplot, axes may not be iterable the same way
    try:
        axes = axes.flatten()
    except Exception:
        axes = [axes]

    for ax in axes:
        ax.axis("off")

    for ax, img_path in zip(axes, page_paths):
        img = Image.open(img_path).convert("RGB")
        ax.imshow(img)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    print(f"Showing images {start+1}–{end} of {len(image_paths)} (page {page})")



# function to count images in given folder
# ----------------------------------------------------------------------------------------------------
def count_images(folder):
    extensions = ("*.jpg", "*.jpeg", "*.png")
    count = 0
    for ext in extensions:
        count += len(glob.glob(os.path.join(folder, ext)))
    return count



# function to get latest training path DIR
# ----------------------------------------------------------------------------------------------------
def get_latest_training_path():
    training_dirs = sorted(
        glob.glob("/content/runs/detect/train*"),
        key=os.path.getmtime
    )

    if not training_dirs:
        raise FileNotFoundError("No YOLO training runs found in runs/detect/")

    train_dir = training_dirs[-1]
    print(f"\nUsing training run from: {train_dir}")
    return train_dir


# function to get latest prediction path DIR
# ----------------------------------------------------------------------------------------------------
def get_latest_prediction_path():
  predict_dirs = sorted(
    glob.glob("runs/detect/predict*"),
    key=os.path.getmtime
  )

  PREDICT_DIR = predict_dirs[-1]
  print(f"\n\nUsing predictions from: {PREDICT_DIR}")
  return PREDICT_DIR


# function to get latest model best.pt path
# ----------------------------------------------------------------------------------------------------
def get_latest_pt_path(latest_train_path):
  pt_files = glob.glob(f"{latest_train_path}/weights/best.pt", recursive=True)

  if not pt_files:
      raise FileNotFoundError("No best.pt file found")

  return pt_files[0]


# function to get list of predicted / inferenced images
# ----------------------------------------------------------------------------------------------------
def get_inferenced_images(predict_dir):
  image_paths = glob.glob(os.path.join(predict_dir, "*.jpg")) + \
              glob.glob(os.path.join(predict_dir, "*.png"))

  return sorted(image_paths)


# function to upload image / video file
# ----------------------------------------------------------------------------------------------------
def upload_image(save_dir="/content/uploads"):

    from pathlib import Path
    import os
    from google.colab import files

    """
    Prompt the user to upload one image and save it to disk.
    Returns the full file path.
    Raises an exception if user cancels.
    """

    folder = Path(save_dir)
    folder.mkdir(parents=True, exist_ok=True)

    uploaded = files.upload()

    # User clicked cancel or uploaded nothing
    if not uploaded:
        raise RuntimeError("Upload cancelled. No file selected.")

    filename = next(iter(uploaded))
    filepath = os.path.join(save_dir, filename)

    with open(filepath, "wb") as f:
        f.write(uploaded[filename])

    return filepath

# Upload Videos
# ----------------------------------------------------------------------------------------------------
def upload_video(save_dir="/content/uploads"):
    from pathlib import Path
    import os
    from google.colab import files

    folder = Path(save_dir)
    folder.mkdir(parents=True, exist_ok=True)

    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No video uploaded.")

    filename = list(uploaded.keys())[0]
    return filename

# function to PRINT GPU Summary
# ----------------------------------------------------------------------------------------------------
def print_gpu_cv_summary(device_index: int = 0) -> None:
    """
    Pretty-print the GPU summary in a readable table-like format.
    """
    result = get_gpu_cv_summary(device_index)
    rows = result["gpu_summary"]

    print(f"\nGPU CV/YOLO Capability Summary (device {result['device_index']})")
    print("-" * 78)
    print(f"{'Property':<28} {'Value':<20} {'Why it matters'}")
    print("-" * 78)
    for r in rows:
        print(f"{r['property']:<28} {str(r['value']):<20} {r['why_it_matters']}")
    print("-" * 78)


# function to determine estimated time of completion
# ----------------------------------------------------------------------------------------------------
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

def estimate_completion_time(minutes_from_now):
    now = datetime.now(ZoneInfo("America/New_York"))
    eta = now + timedelta(minutes=minutes_from_now)

    return {
        "start_time": now.strftime('%Y-%m-%d %H:%M:%S'),
        "estimated_completion": eta.strftime('%Y-%m-%d %H:%M:%S'),
        "minutes_from_now": minutes_from_now
    }


# function to estimate training time required for the model
# ----------------------------------------------------------------------------------------------------
def estimate_training_time(
    num_epochs,
    batch_size,
    train_image_count,
    val_image_count=0,
    batches_per_second=None,
    val_fraction_of_train_time=0.3
):
    """
    Estimate total training time.

    Provide either seconds_per_batch OR seconds_per_epoch (preferrably time for a full epoch).
    If both provided, seconds_per_epoch takes precedence.

    Returns dict with per-epoch and total estimates.
    """


    if batches_per_second is None:
        raise ValueError("Provide speed estimate in Iteractions (batch) per second.")

    seconds_per_batch = 1 / batches_per_second

    train_batches_per_epoch = math.ceil(train_image_count / batch_size)

    train_time_per_epoch = train_batches_per_epoch * seconds_per_batch
    val_time_per_epoch = 0
    if val_image_count > 0:
        # Option: scale validation time by image counts, or use fraction
        # Estimate validation time proportionally to image counts:
        val_batches_per_epoch = math.ceil(val_image_count / batch_size)
        val_time_per_epoch = val_batches_per_epoch * seconds_per_batch * val_fraction_of_train_time

    total_time_seconds = num_epochs * (train_time_per_epoch + val_time_per_epoch)

    return {
        "train_batches_per_epoch": train_batches_per_epoch,
        "train_time_per_epoch_sec": round(train_time_per_epoch, 2),
        "val_time_per_epoch_sec": round(val_time_per_epoch, 2),
        "total_time_sec": round(total_time_seconds, 2),
        "total_time_min": round(total_time_seconds / 60, 2),
        "total_time_hr": round(total_time_seconds / 3600, 2),
        "seconds_per_batch": round(seconds_per_batch, 4),
    }




print(f"""
===== Loaded Helper Functions =====

save_file_to_drive                 :
save_file_to_google_drive          :
show_image_grid_paged              :
count_images                       :
get_latest_training_path           :
get_latest_prediction_path         :
get_latest_pt_path                 :
get_inferenced_images              :

==================================
""")

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ROBOFLOW HELPER FUNCTIONS
#
# Roboflow Metadata Validation and Data Split Management
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++



# HELPERS - METADATA
# ----------------------------------------------------------------------------------------------------
def get_project_metadata(api_key: str, workspace: str, project: str) -> dict:
    """
    Uses Roboflow REST API to retrieve project info and version list.
    Docs show this endpoint returns project images, splits, and versions.
    """
    url = f"https://api.roboflow.com/{workspace}/{project}?api_key={api_key}"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return resp.json()


def get_version_metadata(api_key: str, workspace: str, project: str, version_number: int) -> dict:
    """
    Fetch metadata for a specific dataset version.
    """
    url = f"https://api.roboflow.com/{workspace}/{project}/{version_number}?api_key={api_key}"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return resp.json().get("version", {})


def print_version_summary(version_meta: dict) -> None:
    splits = version_meta.get("splits", {})
    print("\nVersion summary")
    print("-" * 40)
    print(f"Version ID: {version_meta.get('id')}")
    print(f"Version name: {version_meta.get('name')}")
    print(f"Total images: {version_meta.get('images', 0)}")
    print(f"Train images: {splits.get('train', 0)}")
    print(f"Valid images: {splits.get('valid', 0)}")
    print(f"Test images:  {splits.get('test', 0)}")
    print("-" * 40)


# HELPERS: LOCAL DATASET
# ----------------------------------------------------------------------------------------------------
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
LABEL_EXT = ".txt"


def find_images_in_dir(images_dir: Path) -> list[Path]:
    files = []
    for p in images_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            files.append(p)
    return sorted(files)


def corresponding_label_path(image_path: Path, images_root: Path, labels_root: Path) -> Path:
    rel = image_path.relative_to(images_root)
    return (labels_root / rel).with_suffix(LABEL_EXT)


def gather_yolo_pairs(dataset_root: Path) -> list[tuple[Path, Path | None]]:
    """
    Collect all image/label pairs from existing train/valid/test folders.
    Assumes standard YOLO structure:
      dataset_root/
        train/images
        train/labels
        valid/images
        valid/labels
        test/images
        test/labels
    """
    pairs = []

    for split_name in ["train", "valid", "val", "test"]:
        images_root = dataset_root / split_name / "images"
        labels_root = dataset_root / split_name / "labels"
        if not images_root.exists():
            continue

        for image_path in find_images_in_dir(images_root):
            label_path = corresponding_label_path(image_path, images_root, labels_root)
            pairs.append((image_path, label_path if label_path.exists() else None))

    return pairs


def clear_split_dirs(dataset_root: Path) -> None:
    for split in ["train", "valid", "test"]:
        split_dir = dataset_root / split
        if split_dir.exists():
            shutil.rmtree(split_dir)

        (dataset_root / split / "images").mkdir(parents=True, exist_ok=True)
        (dataset_root / split / "labels").mkdir(parents=True, exist_ok=True)


def resplit_dataset(dataset_root: Path, train_ratio: float, val_ratio: float, test_ratio: float) -> dict:
    if round(train_ratio + val_ratio + test_ratio, 6) != 1.0:
        raise ValueError("train_ratio + val_ratio + test_ratio must equal 1.0")

    pairs = gather_yolo_pairs(dataset_root)
    if not pairs:
        raise RuntimeError("No images found in dataset to split.")

    random.seed(RANDOM_SEED)
    random.shuffle(pairs)

    total = len(pairs)
    n_train = int(total * train_ratio)
    n_val = int(total * val_ratio)

    train_pairs = pairs[:n_train]
    val_pairs = pairs[n_train:n_train + n_val]
    test_pairs = pairs[n_train + n_val:]

    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_root = Path(tmp_dir)

        def stage_pairs(pairs_list, split_name: str):
            staged = []
            for i, (image_path, label_path) in enumerate(pairs_list):
                staged_image = tmp_root / f"{split_name}_{i}{image_path.suffix.lower()}"
                shutil.copy2(image_path, staged_image)

                staged_label = None
                if label_path and label_path.exists():
                    staged_label = tmp_root / f"{split_name}_{i}.txt"
                    shutil.copy2(label_path, staged_label)

                staged.append({
                    "image_tmp": staged_image,
                    "label_tmp": staged_label,
                    "image_name": image_path.name,
                    "label_name": f"{image_path.stem}.txt",
                })
            return staged

        staged_train = stage_pairs(train_pairs, "train")
        staged_valid = stage_pairs(val_pairs, "valid")
        staged_test = stage_pairs(test_pairs, "test")

        clear_split_dirs(dataset_root)

        def restore_pairs(staged_pairs, split_name: str):
            for item in staged_pairs:
                target_image = dataset_root / split_name / "images" / item["image_name"]
                shutil.copy2(item["image_tmp"], target_image)

                if item["label_tmp"]:
                    target_label = dataset_root / split_name / "labels" / item["label_name"]
                    shutil.copy2(item["label_tmp"], target_label)

        restore_pairs(staged_train, "train")
        restore_pairs(staged_valid, "valid")
        restore_pairs(staged_test, "test")

    return {
        "total": total,
        "train": len(train_pairs),
        "valid": len(staged_valid),
        "test": len(staged_test),
    }


def update_data_yaml(dataset_root: Path) -> Path:
    """
    Rewrites data.yaml for standard YOLO training.
    """
    yaml_path = dataset_root / "data.yaml"
    if not yaml_path.exists():
        raise FileNotFoundError(f"Could not find {yaml_path}")

    with open(yaml_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f) or {}

    # Preserve class info if present
    data["train"] = str((dataset_root / "train" / "images").resolve())
    data["val"] = str((dataset_root / "valid" / "images").resolve())
    data["test"] = str((dataset_root / "test" / "images").resolve())

    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, sort_keys=False)

    return yaml_path


def validate_roboflow_dataset(
    api_key: str,
    workspace: str,
    project_slug: str,
    dataset_version: int,
    min_train_images: int
) -> dict:
    """
    Fetch Roboflow project/version metadata, print a summary,
    and validate whether the dataset is suitable for download.

    Returns a dictionary with useful metadata if validation passes.
    Raises an exception if validation fails.
    """

    project_meta = get_project_metadata(api_key, workspace, project_slug)

    version_meta = get_version_metadata(
        api_key,
        workspace,
        project_slug,
        dataset_version
    )

    if version_meta is None:
        raise RuntimeError(
            f"Could not find version {dataset_version} "
            f"for {workspace}/{project_slug}"
        )



    version_splits = version_meta.get("splits", {})
    train_count = int(version_splits.get("train", 0))
    valid_count = int(version_splits.get("valid", 0))
    total_count = int(version_meta.get("images", 0))

    has_validation = valid_count > 0

    #print("----------------------------------------")
    #print(f"Valid Images Exist? {has_validation}")

    print_version_summary(version_meta)

    if train_count < min_train_images:
        raise ValueError(
            f"Skipping download: training image count is {train_count}, "
            f"which is below the threshold of {min_train_images}."
        )

    return {
        "project_meta": project_meta,
        "version_meta": version_meta,
        "train_count": train_count,
        "valid_count": valid_count,
        "total_count": total_count,
        "valid_exist": has_validation,
    }


print(f"""
===== Loaded Helper Functions =====

validate_roboflow_dataset          :
show_image_grid_paged              :
count_images                       :
get_latest_training_path           :
get_latest_prediction_path         :
get_latest_pt_path                 :
get_inferenced_images              :

==================================
""")

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Install Ultralytics and Roboflow

# Ultralytics is necessary because it provides the YOLO framework and APIs required to load the .pt model and run object detection inference in Python.
# This .pt file (weights file) is what we will use in our inference scripts
#
# Install and Import Roboflow
#
# Leveraging Roboflow (or even HuggingFace) enables ready-to-use, versioned annotated datasets; This
# simplifies dataset management, preprocessing, and creating reproducible training workflows - without having to staore or manage the data locally
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

!pip install -q roboflow ultralytics

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Install and Import Roboflow

# Leveraging Roboflow (or even HuggingFace) enables ready-to-use, versioned annotated datasets; This
# simplifies dataset management, preprocessing, and creating reproducible training workflows - without having to staore or manage the data locally
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

from roboflow import Roboflow

print(f"""
======= Imported Libraries =======

Roboflow  : Roboflow

==================================
""")

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Prior to Data Importing Dataset, Validate it
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


dataset_info = validate_roboflow_dataset(
    api_key=Roboflow_API_Key,
    workspace=Roboflow_Project_Workspace,
    project_slug=Roboflow_Project_Slug,
    dataset_version=Roboflow_Dataset_Version,
    min_train_images=MIN_TRAIN_IMAGES
)


if (dataset_info["total_count"] <  MIN_TRAIN_IMAGES):
  print(
        f"\nTotal number of images [ {dataset_info["total_count"]} ] , "
        f"is below the threshold minimum of: [ {MIN_TRAIN_IMAGES} ].\n"
    )
  print(figlet_format("STOP"))
  print("----------------------------------------")
  raise ValueError(f"See Error Above")

if (not dataset_info["valid_exist"]):
  #print("----------------------------------------")
  print((f"\nValid Images Not Found.  It is required and dataset will be resplit to establish it."))
  print("----------------------------------------")

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Import the Dataset
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


from pathlib import Path

print(f"Importing Dataset: [{Roboflow_Project_Slug}, Version: [{Roboflow_Dataset_Version}] from: [{Roboflow_Project_Workspace}] (Please wait)\n")


# Initialize Roboflow and create Project Object
rf = Roboflow(api_key=Roboflow_API_Key)
project = rf.workspace(Roboflow_Project_Workspace).project(Roboflow_Project_Slug)
version = project.version(Roboflow_Dataset_Version)

# Clear the existing Dataset if it already exists
dataset_root = Path(DOWNLOAD_DIR) / f"{Roboflow_Project_Slug}-{Roboflow_Dataset_Version}"

# clean target folder only
if dataset_root.exists():
    print("Deleting Dataset: ", dataset_root)
    shutil.rmtree(dataset_root)

dataset_root.mkdir(parents=True, exist_ok=True)

# download dataset
dataset = version.download(
    DOWNLOAD_FORMAT,
    location=str(dataset_root),
    overwrite=True
)

# Store Official Project Name and Folder location to Local Variables
Roboflow_Project_Name = project.name
Roboflow_Project_Folder = dataset.location

# Get Image Count within Training and Validation folders
Data_Train_Count = count_images(os.path.join(Roboflow_Project_Folder, "train", "images"))
Data_Valid_Count = count_images(os.path.join(Roboflow_Project_Folder, "valid", "images"))
Data_Test_Count = count_images(os.path.join(Roboflow_Project_Folder, "test", "images"))


print(f"""
===== Training Data Statistics =====

Roboflow Project Name              : {Roboflow_Project_Name}
Roboflow Project Folder            : {Roboflow_Project_Folder}
Training Data Count                : {Data_Train_Count}
Validation Data Count              : {Data_Valid_Count}
Test Data Count                    : {Data_Test_Count}

====================================
""")


In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Resplit the Dataset if Validation folder is empty according to Split ratios I defined above
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

if Data_Valid_Count < 10:
    print("\nNo validation images found in Roboflow dataset — performing local split...")

    split_result = resplit_dataset(
        dataset_root=dataset_root,
        train_ratio=0.7,
        val_ratio=0.2,
        test_ratio=0.1,
    )

    print("\nLocal split complete")
    print(split_result)

    yaml_path = update_data_yaml(dataset_root)
    print(f"\nUpdated YAML: {yaml_path}")

else:
    print("\nValidation split already exists — skipping local re-split.")

# Get Image Count within Training and Validation folders
Data_Train_Count = count_images(os.path.join(Roboflow_Project_Folder, "train", "images"))
Data_Valid_Count = count_images(os.path.join(Roboflow_Project_Folder, "valid", "images"))
Data_Test_Count = count_images(os.path.join(Roboflow_Project_Folder, "test", "images"))

print(f"""
===== Training Data Statistics =====

Roboflow Project Name              : {Roboflow_Project_Name}
Roboflow Project Folder            : {Roboflow_Project_Folder}
Training Data Count                : {Data_Train_Count}
Validation Data Count              : {Data_Valid_Count}
Test Data Count                    : {Data_Test_Count}

====================================
""")

In [ ]:

# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Get location of the yaml file for this project

# The YAML file defines the dataset configuration for training, including paths to the training/validation
# data and the list of class names, so the YOLO training script knows where the images and labels are
# and how to map class IDs to object categories.
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

yaml_files = glob.glob(f"{Roboflow_Project_Folder}/data.yaml", recursive=True)

if not yaml_files:
    raise FileNotFoundError("No data.yaml file found")

DATA_YAML_PATH = yaml_files[0]

print(f"Using dataset config: {DATA_YAML_PATH}")

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Estimate Time Required to Complete Training
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

estimate = estimate_training_time(
    num_epochs=Config_Epochs,
    batch_size=Config_Batches,
    train_image_count = Data_Train_Count,
    val_image_count = Data_Valid_Count,
    batches_per_second = Config_GPU_Batches_Per_Second,
    val_fraction_of_train_time=0.3
)


# print time estimates
for k, v in estimate.items():
    print(f"{k:<30}: {v}")

# print estimate time of completion
eta_info = estimate_completion_time(estimate["total_time_min"])
for k, v in eta_info.items():
  print(f"{k:<30}: {v}")

print("\n")
print_gpu_cv_summary(0)

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Train model on the data
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


print("Training The Model (Please wait)\n")
print(f"""
===== Training Hyper-parameters =====

Model            : {YOLO_Version}
Model Size       : {YOLO_Size}
Epochs           : {Config_Epochs}
Image Size       : {Config_Image_Size}
Batch Size       : {Config_Batches}
Train Count      : {Data_Train_Count}
Valid Count      : {Data_Valid_Count}

=====================================
""")

#Small dataset	0.001–0.005
#Fine-tuning	0.0005–0.002

from ultralytics import YOLO

DATA_YAML_PATH = DATA_YAML_PATH

model = YOLO("yolov8n.pt")  # small + fast starter model
results = model.train(
    data=DATA_YAML_PATH,
    epochs=Config_Epochs,
    imgsz=Config_Image_Size,
    batch=Config_Batches,
    lr0=Config_Learning_Rate,     # initial LR
    lrf=Config_Learning_Rate      # final LR multiplier (lr_final = lr0 * lrf)

)

# get latest training folder - save it to variabble
LATEST_TRAIN_DIR = get_latest_training_path()


In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Get location of the latest weights generated after training
# The .pt weights file stores the trained YOLO model parameters learned during training, allowing the
# model to perform inference or continue training without retraining from scratch.
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

MODEL_PT_PATH = get_latest_pt_path(LATEST_TRAIN_DIR)

print(f"Using PT File: {MODEL_PT_PATH}")

saved_path = save_file_to_google_drive(
    MODEL_PT_PATH,
    f"/content/drive/MyDrive/training-weights/{Project_Title}.pt"
)

print(f"\nWeights file saved into Google Drive: {saved_path}")

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Run the model against the test data in your project
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

best_model_path = glob.glob(MODEL_PT_PATH)[-1]
model = YOLO(best_model_path)

print(Roboflow_Project_Folder + "/test/images")

model.predict(source = Roboflow_Project_Folder + "/test/images", save=True, conf=0.25)

LATEST_PREDICT_DIR = get_latest_prediction_path()

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Get list of latest predicted / inferenced images
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

image_paths = get_inferenced_images(LATEST_PREDICT_DIR)

print(f"Found {len(image_paths)} predicted images")

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Show images in the Recent Predictions Folder
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

#show first 20 images, 5 per row
show_image_grid_paged(image_paths, cols=8, per_page=24, page=1)

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# View the training plots (Loss Function plots / Accuracy)
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

DisplayImage(filename=f'/{LATEST_TRAIN_DIR}/results.png', width=1000)

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# View the Confusio Matrix
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

DisplayImage(filename=f'/{LATEST_TRAIN_DIR}/confusion_matrix.png', width=1000)

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# View training metrics
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

# model.val(data=DATA_YAML_PATH) # dumps everything (verbose)

metrics = model.val(data=DATA_YAML_PATH)

print("mAP@0.5      :", metrics.box.map50)
print("mAP@0.5:0.95 :", metrics.box.map)
print("Precision    :", metrics.box.mp)
print("Recall       :", metrics.box.mr)


In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Run inference on an arbitrary image and save it to the content folder
#
# alternatively, for images, input could be a URL
# CONFIG_SAMPLE_IMAGE_URL = "https://ufpro.com/Header.jpg"
# model.predict(source=CONFIG_SAMPLE_IMAGE_URL, save=True, conf=0.25)
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

try:
    image_path = upload_image()
except RuntimeError as e:
    print(e)
    image_path = None

if image_path:
  results = model.predict(
      source=image_path,
      save=True,
      conf=0.1,
      name="inferenced",
      exist_ok=True
  )


In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# View the annotated result
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

# find the saved image (YOLO keeps original name or auto-generated one)
# predicted_images = glob.glob(os.path.join("/content/", "*.jpg"))

extensions = (".jpg", ".jpeg", ".png")

pred_dir = "/content/runs/detect/inferenced"

predicted_images = [
    os.path.join(pred_dir, f)
    for f in os.listdir(pred_dir)
    if f.lower().endswith(extensions)
]

# Sort by last modified time (oldest → newest)
predicted_images.sort(key=os.path.getmtime)
latest_image = predicted_images[-1]

print("Opening:", latest_image,"\n")

DisplayImage(filename=latest_image,width=600)

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Run inference on an uploaded video and save it to the content folder
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


try:
    video_path = upload_video()
except RuntimeError as e:
    print(e)
    video_path = None


if video_path:
    results = model.predict(
        source=video_path,
        save=True,
        conf=0.1,
        name="inferenced",
        exist_ok=True
    )

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Install FFMPEG
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

!apt-get -qq install ffmpeg

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Convert the the annotatd AVI into an MP4
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

pred_dir = Path("/content/runs/detect/inferenced")

avi_files = list(pred_dir.glob("*.avi"))

if not avi_files:
    raise RuntimeError("No AVI files found.")

# Get most recently modified file
latest_avi = max(avi_files, key=lambda p: p.stat().st_mtime)

# Get most recently modified file
latest_avi = max(avi_files, key=lambda p: p.stat().st_mtime)

output_mp4 = latest_avi.with_suffix(".mp4")

print("Input:", latest_avi)
print("Output:", output_mp4)

!ffmpeg -i "{latest_avi}" -vcodec libx264 -acodec aac "{output_mp4}" -y


In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# View the the converted MP4 file in an HTML Viewer
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

from IPython.display import HTML
from base64 import b64encode

mp4 = open(output_mp4,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=640 controls>
      <source src="{data_url}" type="video/mp4">
</video>
""")

In [ ]:
#!nvidia-smi -q
#!nvidia-smi